# Shared Spark setup for the streaming jobs.

In [1]:
import os
import glob

os.environ["HADOOP_HOME"] = "D:\\hadoop"
os.environ["PATH"] += os.pathsep + "D:\\hadoop\\bin"

In [2]:
from pathlib import Path
from pyspark.sql import SparkSession, functions as F

In [3]:
jar_files = sorted((Path.cwd() / "jars").glob("*.jar"))
JARS = ",".join(p.resolve().as_posix() for p in jar_files)

CLASSPATH = ";".join(str(p.resolve().as_posix()) for p in jar_files)

In [4]:
print(JARS, CLASSPATH)

D:/Learn Apache Kafka/jars/commons-pool2-2.13.1.jar,D:/Learn Apache Kafka/jars/kafka-clients-3.9.2.jar,D:/Learn Apache Kafka/jars/spark-sql-kafka-0-10_2.13-4.3.0.jar,D:/Learn Apache Kafka/jars/spark-token-provider-kafka-0-10_2.13-4.3.0.jar D:/Learn Apache Kafka/jars/commons-pool2-2.13.1.jar;D:/Learn Apache Kafka/jars/kafka-clients-3.9.2.jar;D:/Learn Apache Kafka/jars/spark-sql-kafka-0-10_2.13-4.3.0.jar;D:/Learn Apache Kafka/jars/spark-token-provider-kafka-0-10_2.13-4.3.0.jar


In [5]:
os.environ["PYSPARK_SUBMIT_ARGS"] = (
    f'--driver-class-path "{CLASSPATH}" --jars "{JARS}" pyspark-shell'
)

In [6]:
import pyspark
print(pyspark.__version__)

4.3.0


In [7]:
ORDER_SCHEMA = ("schema_version INT, order_id STRING, customer_id STRING, product_id STRING, qty INT, price DOUBLE, city STRING, ts STRING")

In [8]:
def get_spark(app_name):
    spark = (SparkSession.builder.appName(app_name)
             .config("spark.jars", JARS)
             .config("spark.driver.extraClassPath", CLASSPATH)
             .config("spark.ui.showConsoleProgress", False)
             .config("spark.sql.shuffle.partitions", "4")
             .getOrCreate()
             )
    spark.sparkContext.setLogLevel("ERROR")
    return spark

In [ ]:
def read_orders(spark):
    raw = (spark.readStream.format("kafka")
           .option("kafka.bootstrap.servers", "localhost:9092")
           .option("subscribe", "orders")
           .option("startingOffsets", "earliest")
           .load()
           )
    good = (F.col("order_id").isNotNull() & F.col("city").isNotNull() & (F.col("qty") >= 1))
    parsed = raw.select(F.from_json(F.col("value").cast("string"), ORDER_SCHEMA).alias("o"),
                        "partition", "offset")
    return (parsed.select("o.*", "partition", "offset")
            .withColumn("event_time", F.to_timestamp("ts"))
            .withColumn("order_date", F.to_date("event_time"))
            .withColumn("revenue", F.round(F.col("qty") * F.col("price"), 2))
            .withColumn("is_valid", F.coalesce(good, F.lit(False)))
            )

In [10]:
# spark = get_spark("spark-common")
# print(spark)

In [11]:
# spark.close()